# 🏗️ 实战挑战：安全帽检测 - 参考答案

**目标**：用安全帽检测数据集训练 YOLO 模型，并诊断类不平衡问题。

---

## 1. 数据集探索

先了解数据集的基本情况：有多少图片、类别分布如何、标注质量如何。

In [ ]:
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import yaml

# 数据集路径
data_dir = Path("data/safety_helmet")
assert data_dir.exists(), "数据集未下载！请先运行: python download.py"

# 读取 data.yaml
with open(data_dir / "data.yaml") as f:
    config = yaml.safe_load(f)

print("数据集配置:")
print(f"  类别数: {config['nc']}")
print(f"  类别名: {config['names']}")

In [ ]:
# 统计图片数量
for split in ['train', 'valid', 'test']:
    img_dir = data_dir / split / 'images'
    lbl_dir = data_dir / split / 'labels'
    if img_dir.exists():
        n_imgs = len(list(img_dir.glob('*')))
        n_lbls = len(list(lbl_dir.glob('*.txt')))
        print(f"  {split}: {n_imgs} 张图片, {n_lbls} 个标注文件")

### 1.1 类别分布分析

**关键问题**：各类别的标注框数量是否均衡？

In [ ]:
# 统计各类别出现次数
class_counts = Counter()
boxes_per_image = []

train_labels = data_dir / 'train' / 'labels'
for lbl_file in train_labels.glob('*.txt'):
    with open(lbl_file) as f:
        lines = [l.strip() for l in f if l.strip()]
        boxes_per_image.append(len(lines))
        for line in lines:
            class_id = int(line.split()[0])
            class_counts[class_id] += 1

names = config['names']
print("各类别标注框数量:")
print(f"{'类别':<15} {'数量':<8} {'占比'}")
print("-" * 35)
total = sum(class_counts.values())
for cls_id in sorted(class_counts.keys()):
    count = class_counts[cls_id]
    pct = count / total * 100
    print(f"{names[cls_id]:<15} {count:<8} {pct:.1f}%")

print(f"\n总计: {total} 个标注框")
print(f"每张图片平均: {sum(boxes_per_image)/len(boxes_per_image):.1f} 个目标")

In [ ]:
# 可视化类分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 类别分布饼图
class_names = [names[i] for i in sorted(class_counts.keys())]
counts = [class_counts[i] for i in sorted(class_counts.keys())]
colors = ['#2ecc71', '#e74c3c', '#3498db']

axes[0].pie(counts, labels=class_names, autopct='%1.1f%%', colors=colors)
axes[0].set_title('类别分布')

# 每张图片目标数分布
axes[1].hist(boxes_per_image, bins=range(0, max(boxes_per_image)+2), edgecolor='black')
axes[1].set_xlabel('每张图片的目标数')
axes[1].set_ylabel('图片数量')
axes[1].set_title('目标数量分布')

plt.tight_layout()
plt.show()

# 诊断
max_count = max(counts)
min_count = min(counts)
ratio = max_count / min_count
print(f"\n⚠️ 类不平衡诊断:")
print(f"  最多类别: {class_names[counts.index(max_count)]} ({max_count})")
print(f"  最少类别: {class_names[counts.index(min_count)]} ({min_count})")
print(f"  比例: {ratio:.1f}:1")
if ratio > 3:
    print(f"  → 类不平衡严重，需要关注")

### 1.2 可视化样本

In [ ]:
from PIL import Image, ImageDraw
import random

# 随机展示几张标注后的图片
train_images = sorted(list((data_dir / 'train' / 'images').glob('*')))
train_labels_list = sorted(list((data_dir / 'train' / 'labels').glob('*.txt')))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
sample_imgs = random.sample(train_images, min(6, len(train_images)))

color_map = {0: '#2ecc71', 1: '#e74c3c', 2: '#3498db'}  # helmet, person, vest

for ax, img_path in zip(axes.flat, sample_imgs):
    img = Image.open(img_path)
    w, h = img.size
    draw = ImageDraw.Draw(img)
    
    # 读取对应标注
    lbl_path = data_dir / 'train' / 'labels' / (img_path.stem + '.txt')
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                cls = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                x1 = (cx - bw/2) * w
                y1 = (cy - bh/2) * h
                x2 = (cx + bw/2) * w
                y2 = (cy + bh/2) * h
                color = color_map.get(cls, 'white')
                draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
                draw.text((x1, y1-12), names.get(cls, str(cls)), fill=color)
    
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 2. 训练模型

用 YOLO11 nano 在安全帽数据集上训练。

In [ ]:
from ultralytics import YOLO

# 加载预训练模型
model = YOLO("yolo11n.pt")

# 训练
results = model.train(
    data=str(data_dir / "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=16,
    device="mps",
    project="runs",
    name="safety_helmet",
    patience=20,
    verbose=True,
)

## 3. 评估模型

In [ ]:
# 加载最佳模型
best_model = YOLO("runs/safety_helmet/weights/best.pt")

# 在验证集上评估
metrics = best_model.val(
    data=str(data_dir / "data.yaml"),
    device="mps",
)

print("=" * 50)
print("评估结果")
print("=" * 50)
print(f"mAP@0.5    : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

In [ ]:
# 各类别 AP 分析
print("各类别 AP@0.5:")
print(f"{'类别':<15} {'AP':<10} {'状态'}")
print("-" * 35)

ap50 = metrics.box.ap50
for i, (cls_id, cls_name) in enumerate(names.items()):
    if i < len(ap50):
        ap = ap50[i]
        status = "✅ 良好" if ap > 0.6 else "⚠️ 需改进" if ap > 0.4 else "❌ 较差"
        print(f"{cls_name:<15} {ap:.4f}    {status}")

## 4. 诊断分析

### 4.1 训练曲线

In [ ]:
import pandas as pd

log_path = Path("runs/safety_helmet/results.csv")
if log_path.exists():
    df = pd.read_csv(log_path, skipinitialspace=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss
    axes[0,0].plot(df['epoch'], df['train/box_loss'], label='box_loss')
    axes[0,0].plot(df['epoch'], df['train/cls_loss'], label='cls_loss')
    axes[0,0].plot(df['epoch'], df['train/dfl_loss'], label='dfl_loss')
    axes[0,0].set_title('Training Loss')
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # mAP
    axes[0,1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5')
    axes[0,1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
    axes[0,1].set_title('Validation mAP')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Precision & Recall
    if 'metrics/precision(B)' in df.columns:
        axes[1,0].plot(df['epoch'], df['metrics/precision(B)'], label='Precision')
        axes[1,0].plot(df['epoch'], df['metrics/recall(B)'], label='Recall')
        axes[1,0].set_title('Precision & Recall')
        axes[1,0].legend()
        axes[1,0].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[1,1].plot(df['epoch'], df['lr/pg0'])
    axes[1,1].set_title('Learning Rate')
    axes[1,1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### 4.2 混淆矩阵

In [ ]:
# 生成混淆矩阵
best_model.val(
    data=str(data_dir / "data.yaml"),
    device="mps",
    plots=True,
)
print("混淆矩阵已保存到 runs/safety_helmet/ 目录")

### 4.3 错误样本分析

找出置信度低或分类错误的样本，分析原因。

In [ ]:
from pathlib import Path

# 在验证集上推理，找出低置信度检测
val_images = sorted(list((data_dir / 'valid' / 'images').glob('*')))

low_conf_samples = []
for img_path in val_images[:50]:  # 检查前 50 张
    results = best_model(str(img_path), conf=0.1, verbose=False)
    for box in results[0].boxes:
        conf = float(box.conf[0])
        if conf < 0.5:
            cls_id = int(box.cls[0])
            low_conf_samples.append({
                'image': img_path.name,
                'class': names.get(cls_id, str(cls_id)),
                'confidence': conf,
            })

if low_conf_samples:
    print(f"发现 {len(low_conf_samples)} 个低置信度检测 (conf < 0.5):")
    for s in low_conf_samples[:10]:
        print(f"  {s['image']}: {s['class']} ({s['confidence']:.2%})")
else:
    print("没有低置信度检测")

## 5. 改进尝试

### 5.1 调整数据增强

针对类不平衡问题，尝试更强的增强策略。

In [ ]:
# 使用更强的增强策略重新训练
model_v2 = YOLO("yolo11n.pt")

results_v2 = model_v2.train(
    data=str(data_dir / "data.yaml"),
    epochs=30,
    imgsz=640,
    batch=16,
    device="mps",
    project="runs",
    name="safety_helmet_v2",
    
    # 增强策略调整
    mosaic=1.0,      # Mosaic 增强
    mixup=0.2,       # MixUp 增强
    hsv_h=0.02,      # 色调增强
    hsv_s=0.8,       # 饱和度增强
    hsv_v=0.5,       # 亮度增强
    degrees=10,      # 旋转
    translate=0.2,   # 平移
    scale=0.8,       # 缩放
)

In [ ]:
# 对比两次训练结果
model_v1 = YOLO("runs/safety_helmet/weights/best.pt")
model_v2_best = YOLO("runs/safety_helmet_v2/weights/best.pt")

metrics_v1 = model_v1.val(data=str(data_dir / "data.yaml"), device="mps")
metrics_v2 = model_v2_best.val(data=str(data_dir / "data.yaml"), device="mps")

print("对比结果:")
print(f"{'指标':<15} {'v1 (默认增强)':<15} {'v2 (强增强)':<15} {'变化'}")
print("-" * 55)
print(f"{'mAP@0.5':<15} {metrics_v1.box.map50:<15.4f} {metrics_v2.box.map50:<15.4f} {metrics_v2.box.map50 - metrics_v1.box.map50:+.4f}")
print(f"{'mAP@0.5:0.95':<15} {metrics_v1.box.map:<15.4f} {metrics_v2.box.map:<15.4f} {metrics_v2.box.map - metrics_v1.box.map:+.4f}")

## 6. 诊断报告模板

完成以上分析后，填写以下报告：

```markdown
### 诊断报告

**数据集**: 安全帽检测 (~500 张, 3 类)

**类分布**:
- person: XXX 个框 (XX%)
- helmet: XXX 个框 (XX%)
- vest: XXX 个框 (XX%)
- 类不平衡比例: X.X:1

**训练结果**:
- mAP@0.5: X.XX
- 最差类别: XXX (AP = X.XX)

**问题诊断**:
1. [问题描述]
2. [原因分析]
3. [改进方案]

**改进效果**:
- 改进前 mAP: X.XX
- 改进后 mAP: X.XX
- 提升: +X.XX
```

---

## 🎯 挑战完成！

**评估你的表现**：

| 等级 | mAP@0.5 | 你做到了吗？ |
|------|---------|-------------|
| 🟡 及格 | > 0.50 | |
| 🟢 良好 | > 0.60 | |
| 🔵 优秀 | > 0.65 | |

**关键收获**：
- 类不平衡会导致少数类 mAP 低
- 数据增强可以缓解类不平衡
- 分析混淆矩阵能快速定位问题
- 调参需要系统性实验，不是盲目试